In [8]:
import re
import json
import xml.etree.ElementTree as ET
from collections import defaultdict
from config import CACHE_DIR, DATASET_DIR
from pathlib import Path
DATASET = DATASET_DIR / "papers"

XLINK = "{http://www.w3.org/1999/xlink}href"

def extract_media_paths(text: str) -> list[str]:
    return re.findall(r'\[\[\[(.+?)\]\]\]', text)

def get_back_media_files(xml_path: Path) -> set[str]:
    root = ET.parse(xml_path).getroot()
    back = root.find(".//back")
    if back is None:
        return set()
    return {
        Path(elem.attrib[XLINK]).name
        for elem in back.iter()
        if XLINK in elem.attrib
    }

def original_name(filename: str) -> str:
    stem = Path(filename).stem  # '717246v1_fig1.tif__0'
    return stem.rsplit("__", 1)[0]  # '717246v1_fig1.tif'

missing_refs = 0
ext_counts = defaultdict(int)

for key_dir in DATASET.iterdir():
    if key_dir.name == ".DS_Store": continue
    key = key_dir.name

    xml_file = list(key_dir.glob('*.xml'))[0]
    pdf_version = f"{xml_file.stem}.pdf"
    media_files = {}
    for file in key_dir.iterdir():
        if file.name not in [xml_file.name, pdf_version] and file.suffix:
            name = original_name(file.name) if file.name.endswith(".png") else file.name
            media_files[name] = True

    preprocess_file = CACHE_DIR / f"{key}.json"
    with open(preprocess_file, "r") as f:
        data = json.load(f)

    if not data['body']:
        print(f"No body: {key_dir}")

    for chunk in data['abstract'] + data['body']:
        for path in extract_media_paths(chunk['text'] + chunk['section']):
            ext_counts[Path(path).suffix.lower()] += 1
            if path not in media_files:
                print(f"Missing: {key}, file: {path}")
                missing_refs += 1

print("Referenced but missing: ", missing_refs)
print("\nExtension counts in preprocessed cache:")
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"  {ext}: {count}")

Referenced but missing:  0

Extension counts in preprocessed cache:
  .tif: 3453
  .gif: 2738
  .pdf: 274
  .jpg: 13
  .tiff: 5
